# Forecasting with the final model (XGBoost)

In [1]:
from xgboost import XGBRegressor
import numpy as np
import pandas as pd

# Loading the final model (XGBoost)
model = XGBRegressor()
model.load_model("../models/xgb_texas_model.json")

# Loading the data for forecasting
df = pd.read_csv("../data/processed/model_data_texas.csv", parse_dates=["Date"])
df = df.set_index("Date").sort_index()
# Raw Texas series (needed for recursive forecasting)
texas_series = df["Texas"].copy().asfreq("MS")

# df.head()

# Target and features
# y = df["target"]
feature_cols = [c for c in df.columns if c not in ["Texas", "target"]]
# X = df[feature_cols]

len(feature_cols), feature_cols


(21,
 ['Texas_lag_1',
  'Texas_lag_2',
  'Texas_lag_3',
  'Texas_lag_6',
  'Texas_lag_12',
  'Texas_lag_24',
  'rolling_3',
  'rolling_6',
  'rolling_12',
  'rolling_12_std',
  'rolling_24',
  'rolling_36',
  'month',
  'quarter',
  'year',
  'is_winter',
  'is_summer',
  'month_sin',
  'month_cos',
  'yoy_change',
  'yoy_24'])

In [2]:
df.index.max(), df.index.min()

(Timestamp('2025-06-01 00:00:00'), Timestamp('2003-12-01 00:00:00'))

In [3]:
#Feature rebuilding functiosn for 12 month forecast horizon
def add_calendar_features(df):
    df = df.copy()
    df["month"] = df.index.month
    df["quarter"] = df.index.quarter
    df["year"] = df.index.year

    df["is_winter"] = df["month"].isin([12, 1, 2]).astype(int)
    df["is_summer"] = df["month"].isin([6, 7, 8]).astype(int)

    df["month_sin"]  = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"]  = np.cos(2 * np.pi * df["month"] / 12)
    return df


def build_features_from_series(series):
    df_fe = pd.DataFrame({"Texas": series}).copy()

    # Lags
    df_fe["Texas_lag_1"] = df_fe["Texas"].shift(1)
    df_fe["Texas_lag_2"] = df_fe["Texas"].shift(2)
    df_fe["Texas_lag_3"] = df_fe["Texas"].shift(3)
    df_fe["Texas_lag_6"] = df_fe["Texas"].shift(6)
    df_fe["Texas_lag_12"] = df_fe["Texas"].shift(12)
    df_fe["Texas_lag_24"] = df_fe["Texas"].shift(24)

    # Rolling windows
    df_fe["rolling_3"] = df_fe["Texas"].rolling(3).mean()
    df_fe["rolling_6"] = df_fe["Texas"].rolling(6).mean()
    df_fe["rolling_12"] = df_fe["Texas"].rolling(12).mean()
    df_fe["rolling_12_std"] = df_fe["Texas"].rolling(12).std()
    df_fe["rolling_24"] = df_fe["Texas"].rolling(24).mean()
    df_fe["rolling_36"] = df_fe["Texas"].rolling(36).mean()

    # YoY features
    df_fe["yoy_change"] = df_fe["Texas"] - df_fe["Texas"].shift(12)
    df_fe["yoy_24"] = df_fe["Texas"] - df_fe["Texas"].shift(24)

    # Add calendar features
    df_fe = add_calendar_features(df_fe)

    return df_fe


In [4]:
# Recursive 12 Month Forecasting Function
from dateutil.relativedelta import relativedelta

def forecast_texas_recursive(series, model, feature_cols, n_steps=12):
    work_series = series.copy().asfreq("MS")
    last_date = work_series.index.max()

    forecasts = []

    for step in range(1, n_steps + 1):
        next_date = last_date + relativedelta(months=1)

        # Extend index to include the new forecast month
        all_idx = work_series.index.union(pd.DatetimeIndex([next_date]))
        work_series = work_series.reindex(all_idx).sort_index()

        # Recompute features on extended series
        df_all = build_features_from_series(work_series)

        # Extract the row for this new month
        X_next = df_all.loc[[next_date], feature_cols]

        # Predict
        y_pred = model.predict(X_next)[0]

        forecasts.append({
            "Date": next_date,
            "forecast": y_pred,
            "horizon": step
        })

        # Insert prediction into series for next iteration
        work_series.loc[next_date] = y_pred
        last_date = next_date

    return pd.DataFrame(forecasts).set_index("Date")


In [5]:
# Run the 12 Month Forecast
df_forecast = forecast_texas_recursive(
    series=texas_series,
    model=model,
    feature_cols=feature_cols,
    n_steps=12
)

df_forecast


,forecast,horizon
Date,,
2025-07-01,443785.06250,1
2025-08-01,451922.75000,2
2025-09-01,420523.25000,3
2025-10-01,418446.75000,4
2025-11-01,387949.65625,5
2025-12-01,429292.81250,6
2026-01-01,457419.06250,7
2026-02-01,414897.46875,8
2026-03-01,383811.21875,9


In [6]:
# Starting from df_forecast: index=Date, cols=['forecast', 'horizon']

df_forecast_out = df_forecast.copy().reset_index()  # Date becomes a column

# Add metadata columns
df_forecast_out["state"] = "Texas"
df_forecast_out["source"] = "forecast"

# Rename forecast → value so it's generic
df_forecast_out = df_forecast_out.rename(columns={"forecast": "value"})

# Optional: reorder columns for readability
df_forecast_out = df_forecast_out[["Date", "state", "value", "source", "horizon"]]

df_forecast_out.head()


,Date,state,value,source,horizon
0,2025-07-01,Texas,443785.06250,forecast,1
1,2025-08-01,Texas,451922.75000,forecast,2
2,2025-09-01,Texas,420523.25000,forecast,3
3,2025-10-01,Texas,418446.75000,forecast,4
4,2025-11-01,Texas,387949.65625,forecast,5


In [7]:
output_path = "../data/processed/texas_forecast_12m.csv"
df_forecast_out.to_csv(output_path, index=False)
output_path


'../data/processed/texas_forecast_12m.csv'